# Tweety C# / IKVM - Semantique propositionnelle : mondes possibles et raisonnement

**Navigation** : [<- Tweety-2-Basic-Logics](Tweety-2-Basic-Logics-Csharp.ipynb) | [Index serie](Tweety-1-Setup.ipynb) | [Tweety-3-Advanced-Logics ->](Tweety-3-Advanced-Logics.ipynb)

> Ce notebook est le **second volet C# / .NET** de la serie Tweety. Il poursuit
> [Tweety-2-Basic-Logics-Csharp](Tweety-2-Basic-Logics-Csharp.ipynb) (construction de formules, parsing, base
> de croyances) en abordant la **semantique** : mondes possibles, satisfaction, et raisonnement
> (entaînement). Comme le pilote, il execute les classes Tweety **recompilees en bytecode .NET via
> IKVM** - aucune JVM au runtime. Voir [le notebook Python](Tweety-2-Basic-Logics.ipynb) pour le contexte theorique complet.

## Objectifs pedagogiques

1. Construire une **signature** propositionnelle (`PlSignature`) et enumerer ses **mondes possibles** (`PossibleWorldIterator`)
2. Tester la **satisfaction** d'une formule par un monde (`PossibleWorld.satisfies`)
3. Effectuer du **raisonnement** : une base de croyances entaîne-t-elle une formule ? (`SatReasoner.query`)
4. Relier l'approche semantique (enumeration des modeles) et l'approche syntaxique (solveur SAT)

## Prerequis

- [Tweety-2-Basic-Logics-Csharp](Tweety-2-Basic-Logics-Csharp.ipynb) (construction de formules, `PlBeliefSet`, `PlParser`)
- Notions de logique propositionnelle : valuation, modele, consequence logique

> **Rappel d'API (C# / IKVM)** : les classes Java Tweety sont exposees via IKVM avec la convention
> Java minuscule (`.add()`, `.size()`, `.hasNext()`...). Le `PlParser.parseFormula()` retourne le type
> de base `commons.Formula` ; on le **cast en `PlFormula`** pour les methodes semantiques.


## 1. Configuration du runtime IKVM

> Identique au [pilote](Tweety-2-Basic-Logics-Csharp.ipynb) : on restaure les paquets NuGet IKVM, on
> assemble le home IKVM (motrice d'execution de la JVM en .NET), puis on charge la DLL Tweety. A executer
> en premier. ~30-60 s la premiere fois.


In [1]:
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "nuget: IKVM.Image.runtime.win-x64, 8.15.0"


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages IKVM, 8.15.0 IKVM.Image, 8.15.0

In [2]:
using System.IO;
string ikvmVer = "8.15.0", ikvmRid = "win-x64";
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome    = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);

void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}

if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
AppContext.SetData("IKVM.Home", ikvmHome);

bool tzdbOk = File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat"));
Console.WriteLine($"IKVM 8.15.0 pret (home=ikvm-home-{ikvmVer}-{ikvmRid}, tzdb={tzdbOk})");


IKVM 8.15.0 pret (home=ikvm-home-8.15.0-win-x64, tzdb=True)


In [3]:
#r "org.tweetyproject.tweety-pl.dll"


In [4]:
// Constat 3 (#5039) : la directive #r est silencieuse par defaut en .NET Interactive.
// On verifie que la DLL IKVM/Tweety referencee est bien presente et lisible (metadata).
using System.Reflection;
var tweetyDll = "org.tweetyproject.tweety-pl.dll";
var an = AssemblyName.GetAssemblyName(tweetyDll);
Console.WriteLine($"Tweety (IKVM) reference chargee : {an.Name} v{an.Version} ({new FileInfo(tweetyDll).Length / 1024 / 1024:F1} Mo).");

Tweety (IKVM) reference chargee : org.tweetyproject.tweety-pl v1.30.0.0 (7,0 Mo).


## 2. Signature et mondes possibles

Une **signature** propositionnelle (`PlSignature`) est l'ensemble des atomes (propositions)
du langage. Pour *n* atomes, il existe **2^n mondes possibles** (valuations) : chaque atome
peut etre vrai ou faux. Un **monde possible** (`PossibleWorld`) est l'ensemble des atomes
consideres comme **vrais** (les autres etant faux par complement).

Le `PossibleWorldIterator` enumerera automatiquement tous les mondes d'une signature.

> Exemple : signature `{a, b}` -> 4 mondes : `[]` (aucun), `[a]`, `[b]`, `[a, b]`.


In [5]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.semantics;
using org.tweetyproject.logics.pl.parser;

var p = new PlParser();
var sig = new PlSignature();
sig.add(p.parseFormula("a"));
sig.add(p.parseFormula("b"));

Console.WriteLine($"Signature : {sig}");

Console.WriteLine("Mondes possibles :");
int n = 0;
var it = new PossibleWorldIterator(sig);
while (it.hasNext())
{
    var world = (PossibleWorld) it.next();
    Console.WriteLine($"  monde {n} = {world}");
    n++;
}
Console.WriteLine($"=> {n} mondes pour 2 atomes (2^2 = 4).");


Signature : [a, b]


Mondes possibles :


  monde 0 = []


  monde 1 = [a]


  monde 2 = [b]


  monde 3 = [a, b]


=> 4 mondes pour 2 atomes (2^2 = 4).


### Interpretation : mondes possibles et semantique

Chaque monde correspond a une **interpretation** (valuation de verite) : `{a}` signifie
*a est vrai, b est faux*. Le monde vide `[]` = tous les atomes faux. L'iteration couvre
exhaustivement les **2^n cas** de la table de verite. Cette enumeration est la base de la
**semantique de Tarski** : une formule est vraie dans un monde si sa valuation la satisfait.


## 3. Satisfaction : un monde est-il modele d'une formule ?

La methode **`PossibleWorld.satisfies(PlFormula)`** repond : ce monde rend-il la formule
vraie ? Une formule est **satisfiable** s'il existe au moins un monde qui la satisfait ; elle
est une **tautologie** si *tous* les mondes la satisfont ; elle est une **contradiction** si
aucun monde ne la satisfait.

> On construit un monde en lui ajoutant des `Proposition` (les atomes vrais), puis on teste
> la satisfaction. Notez le **cast `(PlFormula)`** : `parseFormula` retourne le type generique
> `Formula`, que les methodes semantiques precisent en `PlFormula`.


In [6]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.semantics;
using org.tweetyproject.logics.pl.parser;

var p = new PlParser();

// Un monde ou a=vrai, b=faux
var w = new PossibleWorld();
w.add(new Proposition("a"));
Console.WriteLine($"Monde w = {w}");

var fa   = (PlFormula) p.parseFormula("a");
var fb   = (PlFormula) p.parseFormula("b");
var fAND = (PlFormula) p.parseFormula("a && b");
var fOR  = (PlFormula) p.parseFormula("a || b");
var fIMP = (PlFormula) p.parseFormula("a => b");

Console.WriteLine($"  w satisfait a      = {w.satisfies(fa)}");   // vrai  (a dans w)
Console.WriteLine($"  w satisfait b      = {w.satisfies(fb)}");   // faux  (b absent)
Console.WriteLine($"  w satisfait a && b = {w.satisfies(fAND)}"); // faux  (b absent)
Console.WriteLine($"  w satisfait a || b = {w.satisfies(fOR)}");  // vrai  (a vrai)
Console.WriteLine($"  w satisfait a => b = {w.satisfies(fIMP)}"); // faux  (a vrai, b faux)


Monde w = [a]


  w satisfait a      = True


  w satisfait b      = False


  w satisfait a && b = False


  w satisfait a || b = True


  w satisfait a => b = False


### Representation canonique d'un monde

La methode **`getCompleteConjunction(PlSignature)`** produit la conjonction complete
representant un monde dans une signature donnee (un litteral positif par atome vrai, negatif
par atome faux). C'est la **forme normale disjonctive** canonique d'une valuation.


In [7]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.semantics;
using org.tweetyproject.logics.pl.parser;

var p = new PlParser();
var sig = new PlSignature();
sig.add(p.parseFormula("a"));
sig.add(p.parseFormula("b"));
sig.add(p.parseFormula("c"));

// Monde {a, c} dans la signature {a, b, c}
var w = new PossibleWorld();
w.add(new Proposition("a"));
w.add(new Proposition("c"));
Console.WriteLine($"Monde = {w}");
Console.WriteLine($"Conjonction complete dans {{a,b,c}} = {w.getCompleteConjunction(sig)}");


Monde = [a, c]


Conjonction complete dans {a,b,c} = a&&c&&!b


## 4. Raisonnement : consequence logique (entaînement)

Enumerer les mondes est exact mais couteux (2^n). Pour le **raisonnement** - une base de
croyances `KB` entaîne-t-elle une formule `phi` (note `KB |= phi`) ? - on delegue a un
**solveur SAT** interne (Sat4j) via **`SatReasoner.query(PlBeliefSet, PlFormula)`**.

**Principe** : `KB |= phi` ssi il n'existe aucun modele de KB qui falsifie `phi`. Le solveur
verifie cela sans enumerer exhaustivement. La methode retourne un booleen.

> Exemple : avec `KB = { a, a => b }`, on a `KB |= b` (modus ponens) mais `KB |= c` est faux.


In [8]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.reasoner;
using org.tweetyproject.logics.pl.parser;

var p = new PlParser();
var kb = new PlBeliefSet();
kb.add(p.parseFormula("a"));
kb.add(p.parseFormula("a => b"));
Console.WriteLine($"KB = {kb}");

var r = new SatReasoner();
Console.WriteLine($"KB |= b    ? {r.query(kb, (PlFormula)p.parseFormula("b"))}");      // vrai  : modus ponens
Console.WriteLine($"KB |= c    ? {r.query(kb, (PlFormula)p.parseFormula("c"))}");      // faux  : c non derive
Console.WriteLine($"KB |= a||c ? {r.query(kb, (PlFormula)p.parseFormula("a || c"))}"); // vrai  : a vrai => disjonction
Console.WriteLine($"KB |= !a   ? {r.query(kb, (PlFormula)p.parseFormula("!a"))}");     // faux  : KB contient a


KB = { (a=>b), a }


KB |= b    ? true


KB |= c    ? false


KB |= a||c ? true


KB |= !a   ? false


### Interpretation : les deux faces de la verite

Deux approches equivalentes pour `KB |= phi` :
- **Semantique** (enumerateur) : parcourir tous les modeles de KB et verifier que chacun satisfait `phi`.
- **Syntaxique** (solveur SAT) : tester la (in)satisfiabilite de `KB et non phi` - s'il n'y a pas de modele,
  alors `phi` est consequence.

Le `SatReasoner` emploie la seconde (DPLL/CDCL via Sat4j), bien plus efficace que l'enumeration
des 2^n mondes des que *n* grandit. C'est le coeur des moteurs modernes de raisonnement.


## Exemple guide : modeles d'une base de croyances

Croisons les deux approches : trouvons **les mondes qui satisfont toute la KB** (ses modeles).
Un monde est modele de KB s'il satisfait chaque formule. `PossibleWorld.satisfies` accepte
directement un `PlBeliefSet`.


In [9]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.semantics;
using org.tweetyproject.logics.pl.parser;

var p = new PlParser();
var kb = new PlBeliefSet();
kb.add(p.parseFormula("a => b"));   // si a alors b
kb.add(p.parseFormula("b => c"));   // si b alors c
Console.WriteLine($"KB = {kb}");

// Signature {a, b, c} -> 8 mondes ; on filtre ceux qui sont modeles de KB
var sig = new PlSignature();
sig.add(p.parseFormula("a"));
sig.add(p.parseFormula("b"));
sig.add(p.parseFormula("c"));

Console.WriteLine("Modeles de KB :");
int models = 0;
var it = new PossibleWorldIterator(sig);
while (it.hasNext())
{
    var world = (PossibleWorld) it.next();
    if (world.satisfies(kb))
    {
        Console.WriteLine($"  {world}  -> conjonction complete : {world.getCompleteConjunction(sig)}");
        models++;
    }
}
Console.WriteLine($"=> {models} modele(s) sur 8.");


KB = { (a=>b), (b=>c) }


Modeles de KB :


  []  -> conjonction complete : !a&&!b&&!c


  [c]  -> conjonction complete : c&&!a&&!b


  [b, c]  -> conjonction complete : b&&c&&!a


  [a, b, c]  -> conjonction complete : a&&b&&c


=> 4 modele(s) sur 8.


### Interpretation

KB impose `c` des que `a` est vrai (chainage `a => b => c`). Les modeles respectent donc
l'ordre : si `a` est present, `b` et `c` doivent l'etre aussi. Verifiez sur la liste : aucun
monde ne contient `a` sans `b`, ni `b` sans `c`. C'est exactement la **fermeture transitive**
des implications, vue semantiquement.

> On retrouve ici le lien profond : la consequence logique `KB |= phi` equivaut a dire que
> `phi` est vraie dans **tous** les modeles de KB - c'est le **theorem de Tarski**.


## Conclusion

Ce notebook complete le pilote [Tweety-2-Basic-Logics-Csharp](Tweety-2-Basic-Logics-Csharp.ipynb)
avec la **semantique** de la logique propositionnelle en .NET natif :

| Concept | Classe Tweety (C#) | Methode | Statut |
|---------|-------------------|---------|--------|
| Signature | `PlSignature` | `.add()` | OK |
| Monde possible | `PossibleWorld` | `.add(Proposition)`, `.satisfies()` | OK |
| Enumeration des mondes | `PossibleWorldIterator` | `.hasNext()`, `.next()` | OK |
| Conjonction complete | `PossibleWorld` | `.getCompleteConjunction(sig)` | OK |
| Raisonnement (entaînement) | `SatReasoner` | `.query(PlBeliefSet, PlFormula)` | OK |

**Deux approches equivalentes** : enumeration semantique (2^n) pour la pedagogie, solveur SAT
(Sat4j) pour l'efficacite. Le port .NET couvre desormais la construction **et** le raisonnement
de la logique propositionnelle, sans JVM.

### Limitation

Comme le pilote, ce notebook porte uniquement le **cluster `pl`** (logique propositionnelle).
Les logiques avancees (Description Logic, Modale, QBF - [Tweety-3-Advanced-Logics](Tweety-3-Advanced-Logics.ipynb))
requierent d'autres clusters Tweety (dl, ml) non inclus dans cette DLL ; les notebooks Python restent
canoniques pour ces logiques, ainsi que pour l'argumentation (Dung, notebooks 5-7) et les reseaux
logiques markoviens (notebook 10).

### Exercices

Adaptez ceux du [notebook Python](Tweety-2-Basic-Logics.ipynb) a l'API C# (cast `PlFormula`, methodes
Java minuscules). Rappel : les stubs doivent s'executer sans erreur (jamais `raise`/`assert`/`1/0`).


#### Exercice 1 : Tautologie, contradiction ou contingence ?

Ecrivez une fonction qui, etant donne une formule et une signature, determine si elle est
**tautologie** (tous les mondes la satisfont), **contradiction** (aucun), ou **contingente**.
Testez sur `a || !a`, `a && !a`, et `a => b`.

**Indice** : enumerer les mondes via `PossibleWorldIterator`, compter ceux qui satisfont la formule,
comparer au total `2^n`.


In [10]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.semantics;
using org.tweetyproject.logics.pl.parser;

// TODO etudiant : implementer Classify(p, sig, formulaStr) -> "tautologie"/"contradiction"/"contingente"
string Classify(PlParser p, PlSignature sig, string formulaStr)
{
    var f = (PlFormula) p.parseFormula(formulaStr);
    int total = 0, satisfied = 0;
    // TODO : parcourir PossibleWorldIterator(sig), compter total et les mondes qui satisfont f
    return "Exercice 1 a completer";
}

var p = new PlParser();
var sig = new PlSignature();
sig.add(p.parseFormula("a"));
sig.add(p.parseFormula("b"));
Console.WriteLine(Classify(p, sig, "a || !a"));   // attendu : tautologie
Console.WriteLine(Classify(p, sig, "a && !a"));   // attendu : contradiction
Console.WriteLine(Classify(p, sig, "a => b"));    // attendu : contingente


Exercice 1 a completer


Exercice 1 a completer


Exercice 1 a completer



(9,9): warning CS0219: La variable 'total' est assignée, mais sa valeur n'est jamais utilisée

(9,20): warning CS0219: La variable 'satisfied' est assignée, mais sa valeur n'est jamais utilisée



#### Exercice 2 : Equivalence logique via l'entaînement

Deux formules `f1` et `f2` sont **logiquement equivalentes** ssi `f1 |= f2` ET `f2 |= f1`.
Verifiez que `a => b` est equivalent a `!a || b`, et que `!(a && b)` est equivalent a `!a || !b`
(loi de De Morgan).

**Indice** : construire un `PlBeliefSet` contenant `f1`, puis `SatReasoner.query(kb, f2)` ;
recommencer en inversant. Equivalent ssi les deux sens sont vrais.


In [11]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.reasoner;
using org.tweetyproject.logics.pl.parser;

// TODO etudiant : implementer AreEquivalent(p, f1Str, f2Str) -> bool
bool AreEquivalent(PlParser p, string f1Str, string f2Str)
{
    var f1 = (PlFormula) p.parseFormula(f1Str);
    var f2 = (PlFormula) p.parseFormula(f2Str);
    var r = new SatReasoner();
    // TODO : KB{f1} |= f2  ET  KB{f2} |= f1
    return false;  // TODO etudiant
}

var p = new PlParser();
Console.WriteLine(AreEquivalent(p, "a => b", "!a || b"));        // attendu : True
Console.WriteLine(AreEquivalent(p, "!(a && b)", "!a || !b"));    // attendu : True (De Morgan)
Console.WriteLine(AreEquivalent(p, "a => b", "b => a"));         // attendu : False


False


False


False


#### Exercice 3 : Deduire une consequence cachee

Soit la base `KB = { (a || b) => c, c => d, a }`. Quelles propositions parmi `{a, b, c, d, e}`
sont entaînees par KB ? Utilisez `SatReasoner.query` pour tester chaque proposition et afficher celles
qui sont consequences. Reflechissez ensuite : pourquoi `d` est-il entaîne alors qu'il n'apparait
dans aucune premisse contenant directement `a` ?

**Indice** : `a` vrai + `(a||b)=>c` donne `c` vrai, puis `c=>d` donne `d` vrai. C'est le **chainage avant**
des implications - verifiez-le machinelement.


In [12]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.reasoner;
using org.tweetyproject.logics.pl.parser;

var p = new PlParser();
var kb = new PlBeliefSet();
kb.add(p.parseFormula("(a || b) => c"));
kb.add(p.parseFormula("c => d"));
kb.add(p.parseFormula("a"));

var r = new SatReasoner();
foreach (var atom in new[]{ "a", "b", "c", "d", "e" })
{
    // TODO etudiant : tester r.query(kb, atom) et afficher "KB |= atom : oui/non"
    Console.WriteLine($"KB |= {atom} ? (Exercice 3 a completer)");
}


KB |= a ? (Exercice 3 a completer)


KB |= b ? (Exercice 3 a completer)


KB |= c ? (Exercice 3 a completer)


KB |= d ? (Exercice 3 a completer)


KB |= e ? (Exercice 3 a completer)
